# AI in Dental Education — Corrected, Simplified Analysis Pipeline

## What changed from the original notebook (audit summary)

1. Academic-performance values outside the plausible 0-100% range were
   previously only FLAGGED (printed to an audit table) but never removed
   from the numeric column used in every downstream analysis. This let
   three impossible values (e.g. 306, 435, 448 on a 0-100% scale) sit
   inside the regression and ML feature matrices, where they inflated
   every variance-inflation factor to 30-45 and distorted the fitted
   StandardScaler used by every ML model.
   -> FIXED: out-of-range values are now set to missing at the point of
      cleaning, before any analysis uses the column.

2. Each analysis section handled missing data differently: EFA used
   listwise deletion, the regression dataset used listwise deletion on
   its own subset of columns, and the ML dataset instead IMPUTED missing
   values (median / most-frequent) inside a pipeline. This meant every
   table in the final report was computed on a different N, and the ML
   models were fit on a mix of real and imputed values while the
   statistical models were not.
   -> FIXED: cleaning now happens once, producing a single analysis-ready
      table. Every analysis (reliability, EFA, descriptives, correlation,
      regression, ML, SHAP) runs on that same table and the same N.
      Imputation is no longer used anywhere; only complete cases are
      analysed, consistent with the "keep only the available clean data"
      requirement.

3. Duplicate rows and fully-empty rows were removed early (this part of
   the original code was correct and is kept unchanged).

The pipeline below is organised into clearly labelled steps and can be
run top-to-bottom as a script or pasted into notebook cells.

In [ ]:
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats

import matplotlib.pyplot as plt
import seaborn as sns

import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from factor_analyzer import (
    FactorAnalyzer,
    calculate_kmo,
    calculate_bartlett_sphericity,
)

import shap

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "savefig.dpi": 300,
        "font.size": 11,
        "axes.titlesize": 13,
        "axes.labelsize": 11,
    }
)

## STEP 1 — FILE / OUTPUT CONFIGURATION

In [ ]:
DATA_FILE = Path("data.csv")  # <- point this at your raw CSV
OUTPUT_DIR = DATA_FILE.parent / "AI_Dental_Study_Results_CORRECTED"
FIG_DIR = OUTPUT_DIR / "figures"
TABLE_DIR = OUTPUT_DIR / "tables"
for d in (OUTPUT_DIR, FIG_DIR, TABLE_DIR):
    d.mkdir(exist_ok=True, parents=True)

# A single running log of how many rows survive each cleaning step,
# so the final sample size is fully traceable.
cleaning_log = []


def log_step(step_name, n_rows):
    cleaning_log.append({"Step": step_name, "N_rows": n_rows})
    print(f"[{step_name}] N = {n_rows}")

## STEP 2 — LOAD RAW DATA

In [ ]:
df_raw = pd.read_csv(DATA_FILE)
log_step("Raw file loaded", len(df_raw))

## STEP 3 — CLEAN COLUMN NAMES AND TEXT VALUES

In [ ]:
def clean_column_name(x):
    x = str(x).replace("\ufeff", "").replace("\xa0", " ")
    x = x.replace("\r", " ").replace("\n", " ")
    return re.sub(r"\s+", " ", x).strip()


def clean_text_value(x):
    if pd.isna(x):
        return np.nan
    x = str(x).replace("\xa0", " ")
    return re.sub(r"\s+", " ", x).strip()


df = df_raw.copy()
df.columns = [clean_column_name(c) for c in df.columns]
for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].apply(clean_text_value)

## STEP 4 — REMOVE FULLY-EMPTY ROWS/COLUMNS AND EXACT DUPLICATES

In [ ]:
df = df.dropna(axis=1, how="all")
df = df.dropna(axis=0, how="all")
n_duplicates = int(df.duplicated().sum())
df = df.drop_duplicates().reset_index(drop=True)
log_step(f"After removing {n_duplicates} exact duplicate rows", len(df))

## STEP 5 — IDENTIFY DEMOGRAPHIC / ACADEMIC COLUMNS

In [ ]:
def find_column(columns, patterns):
    for pattern in patterns:
        for col in columns:
            if re.search(pattern, col, flags=re.I):
                return col
    return None


age_col = find_column(df.columns, [r"^age$"])
gender_col = find_column(df.columns, [r"gender", r"sex"])
year_col = find_column(df.columns, [r"year.*study", r"year"])
academic_col = find_column(
    df.columns,
    [r"last.*year.*cgpa", r"academic.*percentage", r"percentage", r"cgpa"],
)
if academic_col is None:
    raise ValueError("Academic percentage/CGPA column could not be identified.")

print("Age column:", age_col)
print("Gender column:", gender_col)
print("Year-of-study column:", year_col)
print("Academic-performance column:", academic_col)

## STEP 6 — PARSE ACADEMIC PERFORMANCE AND REMOVE OUT-OF-RANGE VALUES

In [ ]:
def parse_numeric_academic(x):
    """Convert '74%', '73.50%', '78', etc. to a float. Returns NaN if unparseable."""
    if pd.isna(x):
        return np.nan
    s = str(x).strip().replace(",", "").replace("%", "").strip()
    if s == "":
        return np.nan
    m = re.search(r"[-+]?\d+(?:\.\d+)?", s)
    return float(m.group()) if m else np.nan


df["Academic_Performance_Numeric"] = df[academic_col].apply(parse_numeric_academic)

n_not_recorded = df["Academic_Performance_Numeric"].isna().sum()
out_of_range_mask = (df["Academic_Performance_Numeric"] < 0) | (
    df["Academic_Performance_Numeric"] > 100
)
n_out_of_range = int(out_of_range_mask.sum())

# --- CORRECTED BEHAVIOUR: out-of-range values are removed (set to missing),
#     not merely flagged, since a 0-100% scale cannot contain values such
#     as 306 or 448. All other responses from these participants are kept;
#     only this one variable is affected for these rows.
df.loc[out_of_range_mask, "Academic_Performance_Numeric"] = np.nan

print(f"Academic performance: {n_not_recorded} not recorded, "
      f"{n_out_of_range} out of range and removed")

# Audit table so the exclusion remains fully traceable
academic_audit = df[[academic_col, "Academic_Performance_Numeric"]].copy()
academic_audit["Excluded_out_of_range"] = out_of_range_mask.values
academic_audit.to_excel(OUTPUT_DIR / "Academic_Performance_Cleaning_Audit.xlsx", index=False)

## STEP 7 — STANDARDIZE LIKERT RESPONSES (1-5)

In [ ]:
likert_map = {
    "strongly disagree": 1,
    "disagree": 2,
    "neutral": 3,
    "agree": 4,
    "strongly agree": 5,
}


def normalize_likert(x):
    if pd.isna(x):
        return np.nan
    s = re.sub(r"\s+", " ", str(x).strip().lower())
    return likert_map.get(s, np.nan)


non_questionnaire = {
    c for c in [age_col, gender_col, year_col, academic_col,
                "Academic_Performance_Numeric"] if c is not None
}
question_cols = [c for c in df.columns if c not in non_questionnaire]

for col in question_cols:
    df[col + " [score]"] = df[col].apply(normalize_likert)

## STEP 8 — MAP ITEMS TO DOMAINS AND COMPUTE DOMAIN SCORES

In [ ]:
def find_question_column(patterns, columns=question_cols):
    for col in columns:
        low = col.lower()
        if all(re.search(p, low) for p in patterns):
            return col
    return None


item_map = {
    "AI_Exposure": find_question_column([r"exposed", r"ai tools", r"before"]),
    "AI_Regular_Use": find_question_column([r"use ai tools", r"regularly", r"studying"]),
    "Difficult_Concepts": find_question_column([r"ai helps me understand difficult concepts"]),
    "Note_Preparation": find_question_column([r"ai improves note preparation"]),
    "Exam_Preparation": find_question_column([r"ai assists in examination preparation"]),
    "Visual_Learning": find_question_column([r"ai improves visual learning"]),
    "Diagnosis": find_question_column([r"ai helps me understand diagnosis"]),
    "Treatment_Planning": find_question_column([r"ai improves treatment planning"]),
    "Clinical_Procedures": find_question_column([r"ai helps visualize clinical procedures"]),
    "Radiograph_Interpretation": find_question_column([r"ai improves interpretation of radiographs"]),
    "AI_Diagrams_Teaching": find_question_column([r"ai-generated diagrams improve teaching"]),
    "AI_Examples_Understanding": find_question_column([r"ai-generated examples enhance understanding"]),
    "Faculty_Integration": find_question_column([r"faculty should integrate ai"]),
    "AI_Inaccuracy": find_question_column([r"ai may provide inaccurate information"]),
    "Critical_Thinking_Concern": find_question_column([r"over-reliance", r"critical thinking"]),
    "Ethical_Concern": find_question_column([r"ethical concerns"]),
    "Learning_Efficiency": find_question_column([r"ai improves my learning efficiency"]),
    "Study_Time": find_question_column([r"ai saves study time"]),
    "Confidence": find_question_column([r"ai increases confidence"]),
    "Recommendation": find_question_column([r"recommend ai use"]),
}

missing_items = [k for k, v in item_map.items() if v is None]
if missing_items:
    print("WARNING — items not found in this dataset:", missing_items)

standard_items = pd.DataFrame(index=df.index)
for item_id, original_col in item_map.items():
    if original_col is not None:
        standard_items[item_id] = df[original_col + " [score]"]

domains = {
    "AI_Adoption": ["AI_Exposure", "AI_Regular_Use"],
    "Academic_Learning": ["Difficult_Concepts", "Note_Preparation", "Exam_Preparation", "Visual_Learning"],
    "Clinical_Learning": ["Diagnosis", "Treatment_Planning", "Clinical_Procedures", "Radiograph_Interpretation"],
    "Teaching_Enhancement": ["AI_Diagrams_Teaching", "AI_Examples_Understanding", "Faculty_Integration"],
    "AI_Risk": ["AI_Inaccuracy", "Critical_Thinking_Concern", "Ethical_Concern"],
    "Learning_Efficiency": ["Learning_Efficiency", "Study_Time", "Confidence", "Recommendation"],
}

construct_scores = pd.DataFrame(index=df.index)
for domain, items in domains.items():
    available = [i for i in items if i in standard_items.columns]
    if len(available) >= 2:
        # Row-wise mean of only the items present for that respondent.
        # NOTE: this can mask a partially-missing response as a complete
        # domain score. Because Step 9 below applies listwise deletion on
        # the raw items (not the aggregated domain score), any respondent
        # who did not answer every item in a domain is excluded from the
        # final analysis sample rather than silently averaged over fewer
        # items.
        construct_scores[domain] = standard_items[available].mean(axis=1)

construct_scores["PAILE"] = construct_scores[
    ["Academic_Learning", "Clinical_Learning", "Learning_Efficiency"]
].mean(axis=1)

## STEP 9 — DEFINE ONE FINAL, FULLY-CLEAN ANALYSIS SAMPLE

In [ ]:
# All analyses below (reliability, EFA, descriptives, correlation,
# regression, ML, SHAP) must use the same set of respondents. Rather than
# merging item-level and domain-level scores into a single table (which
# creates a name collision here, since one item and its parent domain are
# both called "Learning_Efficiency"), we compute one completeness mask
# from every variable the study actually uses, and apply that same mask
# to each table separately. Every downstream table therefore has an
# identical N and refers to the same respondents.
required_columns = pd.concat(
    [
        df[[age_col, gender_col, year_col, "Academic_Performance_Numeric"]],
        standard_items,
    ],
    axis=1,
)
n_before_listwise = len(required_columns)
complete_mask = required_columns.notna().all(axis=1)
log_step("After listwise deletion (complete cases only)", int(complete_mask.sum()))

demographics = (
    df.loc[complete_mask, [age_col, gender_col, year_col, "Academic_Performance_Numeric"]]
    .rename(columns={age_col: "Age", gender_col: "Gender", year_col: "Year_of_study",
                      "Academic_Performance_Numeric": "Academic_Performance"})
    .reset_index(drop=True)
)
standard_items = standard_items.loc[complete_mask].reset_index(drop=True)
construct_scores = construct_scores.loc[complete_mask].reset_index(drop=True)

# Convenience combined table for regression/ML steps, where the name
# collision does not matter because only domain-level scores (not the
# raw items) are used as predictors there.
analysis_df = pd.concat([demographics, construct_scores], axis=1)

analysis_df.to_excel(TABLE_DIR / "Analysis_Ready_Dataset.xlsx", index=False)
pd.DataFrame(cleaning_log).to_excel(OUTPUT_DIR / "Cleaning_Log.xlsx", index=False)

item_cols = list(standard_items.columns)
domain_cols = list(domains.keys())

print("\nFinal analysis sample: N =", len(analysis_df))
print(pd.DataFrame(cleaning_log).to_string(index=False))

## STEP 10 — RELIABILITY (CRONBACH'S ALPHA)

In [ ]:
def cronbach_alpha(data):
    k = data.shape[1]
    if k < 2:
        return np.nan
    item_var = data.var(axis=0, ddof=1)
    total_var = data.sum(axis=1).var(ddof=1)
    if total_var == 0:
        return np.nan
    return (k / (k - 1)) * (1 - item_var.sum() / total_var)


reliability_rows = []
for domain, items in domains.items():
    available = [i for i in items if i in standard_items.columns]
    if len(available) < 2:
        continue
    domain_data = standard_items[available]
    alpha = cronbach_alpha(domain_data)
    for item in available:
        reduced = domain_data.drop(columns=item)
        alpha_deleted = cronbach_alpha(reduced) if reduced.shape[1] >= 2 else np.nan
        reliability_rows.append(
            {"Domain": domain, "Item": item, "N_Items": len(available),
             "Cronbach_alpha": alpha, "Alpha_if_item_deleted": alpha_deleted}
        )
reliability_df = pd.DataFrame(reliability_rows)
reliability_df.to_excel(TABLE_DIR / "Reliability_Analysis.xlsx", index=False)

## STEP 11 — EXPLORATORY FACTOR ANALYSIS

In [ ]:
efa_data = standard_items[item_cols]

kmo_all, kmo_model = calculate_kmo(efa_data)
chi_square_value, p_value = calculate_bartlett_sphericity(efa_data)
kmo_bartlett = pd.DataFrame(
    {"Measure": ["KMO", "Bartlett_chi_square", "Bartlett_df", "Bartlett_p"],
     "Value": [kmo_model, chi_square_value, len(item_cols) * (len(item_cols) - 1) / 2, p_value]}
)
kmo_bartlett.to_excel(TABLE_DIR / "EFA_KMO_Bartlett.xlsx", index=False)

fa = FactorAnalyzer(rotation=None)
fa.fit(efa_data)
eigenvalues, _ = fa.get_eigenvalues()
eigen_df = pd.DataFrame({"Factor": np.arange(1, len(eigenvalues) + 1), "Eigenvalue": eigenvalues})
eigen_df.to_excel(TABLE_DIR / "EFA_Eigenvalues.xlsx", index=False)

plt.figure(figsize=(8, 5))
plt.plot(eigen_df["Factor"], eigen_df["Eigenvalue"], marker="o")
plt.axhline(1, linestyle="--")
plt.xlabel("Factor")
plt.ylabel("Eigenvalue")
plt.title("Exploratory Factor Analysis — Scree Plot")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig_EFA_ScreePlot.png", dpi=300)
plt.close()

# Parallel analysis: compare against randomly permuted data of the same shape
n_iter = 100
random_eigs = np.zeros((n_iter, len(item_cols)))
rng = np.random.default_rng(RANDOM_STATE)
for i in range(n_iter):
    random_data = pd.DataFrame(
        {c: rng.permutation(efa_data[c].values) for c in efa_data.columns}
    )
    fa_r = FactorAnalyzer(rotation=None)
    fa_r.fit(random_data)
    random_eigs[i, :], _ = fa_r.get_eigenvalues()

mean_random_eig = random_eigs.mean(axis=0)
n_retain = int(np.sum(eigenvalues > mean_random_eig))
parallel_df = pd.DataFrame(
    {"Factor": np.arange(1, len(item_cols) + 1),
     "Observed_Eigenvalue": eigenvalues,
     "Mean_Random_Eigenvalue": mean_random_eig,
     "Retain": eigenvalues > mean_random_eig}
)
parallel_df.to_excel(TABLE_DIR / "EFA_Parallel.xlsx", index=False)
print(f"Parallel analysis retains {n_retain} factor(s).")

fa_final = FactorAnalyzer(n_factors=max(n_retain, 1), rotation="varimax")
fa_final.fit(efa_data)
loadings = pd.DataFrame(
    fa_final.loadings_,
    index=item_cols,
    columns=[f"Factor_{i+1}" for i in range(max(n_retain, 1))],
)
loadings["Communality"] = fa_final.get_communalities()
loadings.to_excel(TABLE_DIR / "EFA_Loadings.xlsx")

## STEP 12 — DESCRIPTIVE STATISTICS AND CORRELATIONS

In [ ]:
desc = analysis_df[domain_cols + ["PAILE"]].describe().T
ci = stats.t.interval(
    0.95, df=len(analysis_df) - 1,
    loc=desc["mean"], scale=desc["std"] / np.sqrt(len(analysis_df))
)
desc["95CI_Lower"], desc["95CI_Upper"] = ci
desc.to_excel(TABLE_DIR / "Descriptive_Statistics.xlsx")

corr = analysis_df[domain_cols + ["PAILE"]].corr(method="spearman")
corr.to_excel(TABLE_DIR / "Correlations.xlsx")

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, square=True)
plt.title("Spearman Correlation Matrix of AI-Education Constructs")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig_Correlation_Heatmap.png", dpi=300)
plt.close()

## STEP 13 — REGRESSION (SAME analysis_df; NO SEPARATE CLEANING)

In [ ]:
reg_df = analysis_df[["PAILE", "AI_Adoption", "Teaching_Enhancement", "AI_Risk",
                       "Age", "Academic_Performance", "Gender", "Year_of_study"]].copy()
reg_df["Age"] = pd.to_numeric(reg_df["Age"], errors="coerce")
reg_df = pd.concat(
    [reg_df.drop(columns=["Gender", "Year_of_study"]),
     pd.get_dummies(reg_df["Gender"], prefix="Gender", drop_first=True, dtype=float),
     pd.get_dummies(reg_df["Year_of_study"], prefix="Year", drop_first=True, dtype=float)],
    axis=1,
)

X_reg = reg_df.drop(columns="PAILE").astype(float)
y_reg = reg_df["PAILE"].astype(float)
X_sm = sm.add_constant(X_reg)
ols_model = sm.OLS(y_reg, X_sm).fit()
print(ols_model.summary())

regression_results = pd.DataFrame(
    {"Coefficient": ols_model.params, "SE": ols_model.bse, "t": ols_model.tvalues,
     "p": ols_model.pvalues, "CI_Lower": ols_model.conf_int()[0], "CI_Upper": ols_model.conf_int()[1]}
)
regression_results.to_excel(TABLE_DIR / "OLS_Regression_Results.xlsx")

vif_df = pd.DataFrame(
    {"Variable": X_reg.columns,
     "VIF": [variance_inflation_factor(X_reg.values, i) for i in range(X_reg.shape[1])]}
)
vif_df.to_excel(TABLE_DIR / "Regression_VIF.xlsx", index=False)

## STEP 14 — MACHINE-LEARNING PREDICTION (SAME analysis_df; NO IMPUTATION)

In [ ]:
# Because analysis_df already contains only complete cases (Step 9), the
# preprocessing pipeline below no longer needs a SimpleImputer step —
# there is nothing left to impute. This removes the risk of ML models
# being fit on a mix of real and imputed values while the statistical
# models are fit on real values only.
ml_df = analysis_df[["PAILE", "Age", "Academic_Performance", "AI_Adoption",
                      "Teaching_Enhancement", "AI_Risk", "Gender", "Year_of_study"]].copy()

X = ml_df.drop(columns="PAILE")
y = ml_df["PAILE"].astype(float)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

numeric_features = ["Age", "Academic_Performance", "AI_Adoption", "Teaching_Enhancement", "AI_Risk"]
categorical_features = ["Gender", "Year_of_study"]

preprocessor = ColumnTransformer(
    [
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

models = {
    "Linear Regression": Pipeline([("prep", preprocessor), ("model", LinearRegression())]),
    "Ridge": Pipeline([("prep", preprocessor), ("model", Ridge())]),
    "LASSO": Pipeline([("prep", preprocessor), ("model", Lasso(max_iter=20000))]),
    "SVR": Pipeline([("prep", preprocessor), ("model", SVR())]),
    "Random Forest": Pipeline(
        [("prep", preprocessor),
         ("model", RandomForestRegressor(n_estimators=500, random_state=RANDOM_STATE, n_jobs=-1))]
    ),
}
try:
    from xgboost import XGBRegressor
    models["XGBoost"] = Pipeline(
        [("prep", preprocessor),
         ("model", XGBRegressor(n_estimators=400, max_depth=3, learning_rate=0.03,
                                 subsample=0.8, colsample_bytree=0.8,
                                 objective="reg:squarederror", random_state=RANDOM_STATE, n_jobs=-1))]
    )
except ImportError:
    print("XGBoost not installed; continuing without it.")

cv = KFold(n_splits=10, shuffle=True, random_state=RANDOM_STATE)
ml_results, fitted_models, predictions = [], {}, {}

for name, model in models.items():
    print("Training:", name)
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="r2")
    model.fit(X_train, y_train)
    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    ml_results.append(
        {"Model": name, "CV_R2_Mean": cv_scores.mean(), "CV_R2_SD": cv_scores.std(),
         "Train_R2": r2_score(y_train, train_pred), "Test_R2": r2_score(y_test, test_pred),
         "Test_MAE": mean_absolute_error(y_test, test_pred),
         "Test_RMSE": np.sqrt(mean_squared_error(y_test, test_pred))}
    )
    fitted_models[name] = model
    predictions[name] = test_pred

ml_results_df = pd.DataFrame(ml_results).sort_values("Test_R2", ascending=False)
ml_results_df.to_excel(TABLE_DIR / "ML_Performance.xlsx", index=False)
print(ml_results_df)

plt.figure(figsize=(9, 6))
plot_df = ml_results_df.sort_values("Test_RMSE")
sns.barplot(data=plot_df, x="Test_RMSE", y="Model")
plt.xlabel("Independent Test RMSE")
plt.title("Machine-Learning Model Comparison")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig_ML_Model_Comparison.png", dpi=300)
plt.close()

best_model_name = ml_results_df.iloc[0]["Model"]
best_model = fitted_models[best_model_name]
best_predictions = predictions[best_model_name]
print("Best model (by test R2):", best_model_name)

plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_predictions, alpha=0.75)
lims = [min(y_test.min(), best_predictions.min()), max(y_test.max(), best_predictions.max())]
plt.plot(lims, lims, linestyle="--")
plt.xlabel("Observed PAILE")
plt.ylabel("Predicted PAILE")
plt.title(f"Observed vs Predicted PAILE — {best_model_name}")
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig_Observed_vs_Predicted.png", dpi=300)
plt.close()

## STEP 15 — SHAP EXPLAINABILITY

In [ ]:
preprocessor_fitted = best_model.named_steps["prep"]
X_test_transformed = preprocessor_fitted.transform(X_test)
feature_names = (
    numeric_features
    + list(preprocessor_fitted.named_transformers_["cat"].get_feature_names_out(categorical_features))
)

explainer = shap.Explainer(best_model.named_steps["model"], preprocessor_fitted.transform(X_train))
shap_values = explainer(X_test_transformed, check_additivity=False)

shap_importance = pd.DataFrame(
    {"Feature": feature_names, "Mean_Absolute_SHAP": np.abs(shap_values.values).mean(axis=0)}
).sort_values("Mean_Absolute_SHAP", ascending=False)
shap_importance.to_excel(TABLE_DIR / "SHAP_Importance.xlsx", index=False)

plt.figure()
shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names, plot_type="bar", show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig_SHAP_Global_Importance.png", dpi=300)
plt.close()

plt.figure()
shap.summary_plot(shap_values, X_test_transformed, feature_names=feature_names, show=False)
plt.tight_layout()
plt.savefig(FIG_DIR / "Fig_SHAP_Summary_Beeswarm.png", dpi=300)
plt.close()

print("\nDone. Final analysis N =", len(analysis_df),
      "| Best model:", best_model_name,
      "| Test R2:", round(ml_results_df.iloc[0]["Test_R2"], 3))